# Running Palace Simulations

[Palace](https://awslabs.github.io/palace/) is an open-source 3D electromagnetic simulator supporting eigenmode, driven (S-parameter), and electrostatic simulations. This notebook demonstrates using the `gsim.palace` API to run a driven simulation on custom RF components—specifically, the new spiral inductor generated using `gdsfactory`.

**Requirements:**
- `gsim` with Palace backend
- `circulax` (for lumped-element modeling and data fitting)

In [ ]:
import gdsfactory as gf
from gdsfactory.components.analog.inductors import spiral_inductor

gf.gpdk.PDK.activate()

### Inductor layout

In [ ]:
spiral_inductor().plot()

In [ ]:
cc = spiral_inductor(add_pgs=True)
cc.plot()

### Configure and run simulation with DrivenSim

In [ ]:
from gsim.palace import DrivenSim

# Create simulation object
sim = DrivenSim()

# Set output directory
sim.set_output_dir("./palace-sim-spiral_inductor")

# Set the component geometry
sim.set_geometry(cc)

# Configure layer stack from active PDK
sim.set_stack(substrate_thickness=180.0, include_substrate=True)

# Configure ports
sim.add_port("P1", from_layer="metal1", to_layer="metal3", geometry="via", excited=True)
sim.add_port("P2", from_layer="metal1", to_layer="metal2", geometry="via", excited=True)

# Configure driven simulation (frequency sweep for S-parameters)
sim.set_driven(fmin=10e9, fmax=150e9, num_points=50)

# Validate configuration
print(sim.validate_config())

In [ ]:
# Generate mesh (presets: "coarse", "default", "fine")
sim.set_airbox(margin_x=50, margin_y=50, z_above=50, z_below=5)
sim.mesh(preset="default", refined_mesh_size=3.0, auto_size=True)
sim.write_config()

In [ ]:
sim.plot_mesh(show_groups=["metal", "via", "P"], interactive=True)

In [ ]:
sim.plot_mesh(
    style="solid",
    transparent_groups=["air__None", "sio2__None", "air__sio2"],
)

### Run simulation on cloud

In [ ]:
import palacetoolkit.verify_topology as pt_verify

# 1. Define the paths where your script just saved the mesh and config
mesh_file = "palace-sim-spiral_inductor/palace.msh"
config_file = "palace-sim-spiral_inductor/config.json"

print("Running PalaceToolkit topology verification...")

# 2. Pass the files into the gatekeeper function
verification_results = pt_verify.verify(mesh_path=mesh_file, config_path=config_file)

# 3. Print the formatted report so you can read it
pt_verify.print_report(verification_results)

# Optional: Stop execution if it found non-manifold or duplicate faces
# (You can inspect the exact keys inside verification_results)
if verification_results.get("nonmanifold_faces") or verification_results.get(
    "duplicate_faces"
):
    raise RuntimeError("Topology verification failed! Do not run the solver yet.")

In [ ]:
# Run simulation on GDSFactory+ cloud
results = sim.run()
# results = sim.run_local(palace_executable = "~/palace/build/bin/palace")

In [ ]:
results.plot_interactive()

In [ ]:
results.plot_interactive(phase=True)

In [ ]:
results.plot()